In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam

In [ ]:
def load_and_prepare_data(file_path, window_size=60, feature_cols=['Close'], target_col='Close', split_ratio=0.8):
    """
    Loads data, cleans it, splits it to prevent leakage, and then scales.
    """
    # 1. Load and Sort
    df = pd.read_csv(file_path)
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values('Date')
    
    # 2. Basic Cleaning: Drop missing values (crucial for financial data)
    df = df.dropna(subset=feature_cols + [target_col])
    
    # 3. Split data BEFORE scaling to prevent Data Leakage
    split_index = int(len(df) * split_ratio)
    train_df = df.iloc[:split_index]
    test_df = df.iloc[split_index:]
    
    # 4. Initialize and Fit Scaler only on training data
    scaler = MinMaxScaler(feature_range=(0, 1))
    # Note: If predicting only 'Close', we still fit on the feature set
    scaler.fit(train_df[feature_cols])
    
    # Transform both sets using training parameters
    train_scaled = scaler.transform(train_df[feature_cols])
    test_scaled = scaler.transform(test_df[feature_cols])
    
    # 5. Helper function to create sliding windows
    def create_sequences(data, window):
        X, y = [], []
        for i in range(window, len(data)):
            # Features: last 'window' days
            X.append(data[i-window:i])
            # Target: current day (assuming index 0 is 'Close' or target)
            y.append(data[i, 0]) 
        return np.array(X), np.array(y)

    X_train, y_train = create_sequences(train_scaled, window_size)
    X_test, y_test = create_sequences(test_scaled, window_size)
    
    # 6. Reshape for LSTM [samples, time_steps, features]
    # This automatically detects if you added more features
    n_features = len(feature_cols)
    X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], n_features))
    X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], n_features))
    
    return X_train, y_train, X_test, y_test, scaler

In [ ]:
# def split_data(X, y, split_ratio=0.8):
#     """Splits data chronologically into train and test sets."""
#     split_index = int(len(X) * split_ratio)
    
#     X_train, X_test = X[:split_index], X[split_index:]
#     y_train, y_test = y[:split_index], y[split_index:]
    
#     return X_train, X_test, y_train, y_test

In [ ]:
def build_lstm_model(input_shape):
    """Refined LSTM architecture with Batch Normalization and increased capacity."""
    model = Sequential([
        # Layer 1: Increased units and added BatchNormalization
        LSTM(units=100, return_sequences=True, input_shape=input_shape),
        BatchNormalization(),
        Dropout(0.3),
        
        # Layer 2: Middle layer to capture deeper temporal relationships
        LSTM(units=100, return_sequences=True),
        BatchNormalization(),
        Dropout(0.3),
        
        # Layer 3: Final LSTM layer (return_sequences=False)
        LSTM(units=50, return_sequences=False),
        BatchNormalization(),
        Dropout(0.2),
        
        # Output Layer: Dense layer for the single price prediction
        Dense(units=25, activation='relu'),
        Dense(units=1)
    ])
    
    # Using a slightly lower learning rate for more stable training
    optimizer = Adam(learning_rate=0.001)
    
    model.compile(optimizer=optimizer, loss='mean_squared_error')
    return model

In [ ]:
def train_lstm(model, X_train, y_train, epochs=20, batch_size=32):
    """Trains the model over multiple epochs."""
    print(f"Starting training for {epochs} epochs...")
    history = model.fit(
        X_train, y_train, 
        epochs=epochs, 
        batch_size=batch_size, 
        validation_split=0.1, # Use 10% of training data to check errors during training
        verbose=1
    )
    return history

In [ ]:
def evaluate_lstm(model, X_test, y_test, scaler):
    """Predicts and converts values back to original dollar prices."""
    predictions = model.predict(X_test)
    
    # Undo scaling to get actual dollar prices
    predictions = scaler.inverse_transform(predictions)
    y_test_actual = scaler.inverse_transform(y_test.reshape(-1, 1))
    
    mae = mean_absolute_error(y_test_actual, predictions)
    r2 = r2_score(y_test_actual, predictions)
    
    print(f"\nLSTM Model Evaluation:")
    print(f"- Mean Absolute Error: ${mae:.2f}")
    print(f"- R-squared Score: {r2:.4f}")
    
    return y_test_actual, predictions

In [ ]:
def plot_lstm_results(actual, predicted):
    """Visualizes the final predictions vs actual prices."""
    plt.figure(figsize=(12, 6))
    plt.plot(actual[-200:], label='Actual Stock Price', color='blue')
    plt.plot(predicted[-200:], label='Predicted Stock Price', color='red', linestyle='--')
    plt.title('LSTM Prediction: Actual vs Predicted (Last 200 Days)')
    plt.xlabel('Time')
    plt.ylabel('Price (USD)')
    plt.legend()
    plt.show()

In [ ]:
if __name__ == "__main__":
    FILE = 'cleaned.csv'
    WINDOW = 60 # look back at the last 60 days to predict tomorrow
    
    # Updated Usage
    X_train, y_train, X_test, y_test, scaler = load_and_prepare_data('cleaned.csv', window_size=60)

    # Build model using the shape of X_train
    lstm_model = build_lstm_model((X_train.shape[1], X_train.shape[2]))
    train_lstm(lstm_model, X_train, y_train, epochs=10, batch_size=64)
    
    # evaluating and visualizing the results
    actual_prices, predicted_prices = evaluate_lstm(lstm_model, X_test, y_test, scaler)
    plot_lstm_results(actual_prices, predicted_prices)